# Databricks/PySpark dla Data Analyst/Engineer — Moduł 2: DataFrame API — podstawy

W tym module przechodzimy przez fundamentalne operacje na DataFrame'ach: jak je
tworzyć, jak wybierać i przekształcać kolumny, jak filtrować wiersze i jak sortować
wynik. Jeśli znasz `pandas` albo SQL z poprzednich kursów, większość z tego wyda Ci się
znajoma — celowo.

## Spis treści
1. [Tworzenie DataFrame'ów](#sec1)
2. [Jawny schemat](#sec2)
3. [Wybieranie i przekształcanie kolumn](#sec3)
4. [Filtrowanie wierszy](#sec4)
5. [Dodawanie i usuwanie kolumn](#sec5)
6. [Sortowanie](#sec6)
7. [Konwersja do/z pandas](#sec7)
8. [Podsumowanie i ćwiczenia](#sec8)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("kurs_spark").getOrCreate()
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. Tworzenie DataFrame'ów

DataFrame Sparka można stworzyć na kilka sposobów — już poznaliśmy jeden (`createDataFrame`
z pandas DataFrame w Module 1). Oto najczęstsze:

In [ ]:
# Z listy krotek + nazwy kolumn
maly_df = spark.createDataFrame(
    [(1, "Kubek", 19.99), (2, "Talerz", 12.50)],
    ["id", "nazwa", "cena"],
)
maly_df.show()


In [ ]:
# spark.range() -- szybki sposob na wygenerowanie ciagu liczb (przydatne do testow)
zakres_df = spark.range(5)
zakres_df.show()


> ⚡ **createDataFrame z pandas kontra wczytanie pliku**
>
> W Module 1 stworzyliśmy nasze dane przez `spark.createDataFrame(pandas_df)` — wygodne do nauki i małych zbiorów. W prawdziwej pracy dane najczęściej wczytuje się bezpośrednio z plików (`spark.read.csv(...)`, `spark.read.parquet(...)`) — tym zajmiemy się szczegółowo w Module 6.

<a id="sec2"></a>
## 2. Jawny schemat

Kiedy tworzysz DataFrame z surowych danych (np. z pliku CSV bez nagłówka typów), Spark
może błędnie odgadnąć typy kolumn. Można podać schemat JAWNIE przez `StructType`:

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

schemat = StructType([
    StructField("id", IntegerType(), nullable=False),
    StructField("nazwa", StringType(), nullable=True),
    StructField("cena", DoubleType(), nullable=True),
])

produkty_jawny_schemat = spark.createDataFrame(
    [(1, "Kubek", 19.99), (2, "Talerz", 12.50)], schema=schemat
)
produkty_jawny_schemat.printSchema()


> ⚠️ **Wnioskowanie schematu (inferSchema) ma swoją cenę**
>
> Kiedy Spark sam odgaduje typy (np. przy wczytywaniu CSV z `inferSchema=True`), musi PRZECZYTAĆ dane dwa razy — raz, żeby zgadnąć typy, drugi raz, żeby je faktycznie wczytać. Przy dużych zbiorach to zauważalny koszt. Jawny schemat pozwala uniknąć tego podwójnego odczytu — wracamy do tego w Module 6.

<a id="sec3"></a>
## 3. Wybieranie i przekształcanie kolumn

`select()` wybiera podzbiór kolumn — dokładnie jak `SELECT` w SQL-u. Kolumny można
odwoływać się na kilka równoważnych sposobów: przez nazwę w cudzysłowie, przez
`df.nazwa_kolumny`, albo przez `F.col("nazwa_kolumny")` (ten ostatni jest najbardziej
elastyczny, bo pozwala budować wyrażenia).

In [ ]:
from pyspark.sql import functions as F

klienci.select("klient_id", "miasto").show(5)


In [ ]:
# Wyrazenie kolumnowe + alias -- tworzymy nowa, obliczona kolumne w locie
klienci.select(
    "klient_id",
    F.col("miasto"),
    (F.col("dni_od_rejestracji") / 365).alias("lata_jako_klient"),
).show(5)


> ⚡ **col() kontra selectExpr() kontra napis SQL**
>
> Te same przekształcenia można zapisać na kilka sposobów: `F.col("x") + 1`, albo `df.selectExpr("x + 1 AS y")` (składnia jak w SQL, jednym stringiem), albo w ogóle czystym Spark SQL (Moduł 5). Wszystkie kompilują się do tego samego planu wykonania — wybierz ten, który czyta się najwygodniej w danym miejscu.

<a id="sec4"></a>
## 4. Filtrowanie wierszy

`filter()` (albo jego alias `where()`) odpowiada `WHERE` z SQL-a. Warunki łączy się
operatorami `&` (i), `|` (lub), `~` (negacja) — UWAGA: nie `and`/`or`/`not` z czystego
Pythona, bo te działają na pojedynczych wartościach logicznych, a nie na całych
kolumnach.

In [ ]:
klienci.filter(F.col("segment") == "Premium").show(5)


In [ ]:
# Warunek zlozony -- pamietaj o nawiasach wokol kazdego podwarunku!
klienci.filter((F.col("segment") == "Premium") & (F.col("miasto") == "Warszawa")).show(5)


> ⚠️ **Nawiasy są obowiązkowe przy złożonych warunkach**
>
> `F.col("a") == 1 & F.col("b") == 2` bez nawiasów zadziała błędnie (albo w ogóle się nie wykona) — Python inaczej ustala priorytet operatorów `&`/`|` niż byśmy się spodziewali intuicyjnie. Zawsze otaczaj KAŻDY podwarunek osobnym nawiasem, jak w przykładzie powyżej.

<a id="sec5"></a>
## 5. Dodawanie i usuwanie kolumn

`withColumn()` dodaje nową kolumnę (albo NADPISUJE istniejącą, jeśli podasz tę samą
nazwę). `withColumnRenamed()` zmienia nazwę. `drop()` usuwa kolumnę.

In [ ]:
zamowienia_z_flaga = zamowienia.withColumn(
    "jest_anulowane", F.col("status") == "Anulowane"
)
zamowienia_z_flaga.show(5)


In [ ]:
zamowienia_zmieniona_nazwa = zamowienia.withColumnRenamed("data_zamowienia", "data")
zamowienia_bez_pracownika = zamowienia.drop("pracownik_id")
zamowienia_bez_pracownika.printSchema()


<a id="sec6"></a>
## 6. Sortowanie

`orderBy()` (albo `sort()`, to synonimy) sortuje wiersze — domyślnie rosnąco, `.desc()`
na kolumnie odwraca kierunek.

In [ ]:
produkty.orderBy(F.col("cena").desc()).show(5)


In [ ]:
# Sortowanie po wielu kolumnach naraz
produkty.orderBy(F.col("kategoria"), F.col("cena").desc()).show(10)


<a id="sec7"></a>
## 7. Konwersja do/z pandas

`.toPandas()` ściąga CAŁY DataFrame Sparka do pamięci Drivera jako zwykły `pandas.DataFrame`
— przydatne na końcu analizy (np. żeby narysować wykres `matplotlib`), ale niebezpieczne
przy dużych danych.

In [ ]:
produkty_pd = produkty.limit(5).toPandas()
print(type(produkty_pd))
produkty_pd


> ⚠️ **toPandas() na dużym DataFrame może zapchać pamięć Drivera**
>
> `.toPandas()` ściąga WSZYSTKIE wiersze do jednej maszyny (Drivera) — jeśli DataFrame ma miliard wierszy, prawdopodobnie nie zmieści się w pamięci Drivera i aplikacja się wywali. Zawsze najpierw ZREDUKUJ dane w Sparku (filtr, agregacja, `.limit()`) i dopiero WYNIK (już mały) konwertuj do pandas.

<a id="sec8"></a>
## 8. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- kilka sposobów tworzenia DataFrame'ów (`createDataFrame`, `spark.range()`),
- jawne definiowanie schematu przez `StructType`/`StructField`,
- wybieranie i przekształcanie kolumn (`select`, `F.col()`, aliasy),
- filtrowanie wierszy (`filter`/`where`, operatory `&`/`|`/`~`),
- dodawanie, zmianę nazwy i usuwanie kolumn (`withColumn`, `withColumnRenamed`, `drop`),
- sortowanie (`orderBy`),
- konwersję do/z `pandas` (`.toPandas()`) i jej pułapkę pamięciową.

### 📝 Ćwiczenie 1: Wybór i przekształcenie kolumn

Z DataFrame'u `produkty` wybierz kolumny `produkt_id`, `kategoria` oraz nową kolumnę `cena_z_vat` (cena razy `1.23`).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
produkty.select(
    "produkt_id",
    "kategoria",
    (F.col("cena") * 1.23).alias("cena_z_vat"),
).show(5)
```

</details>

### 📝 Ćwiczenie 2: Filtrowanie zamówień

Wyfiltruj `zamowienia` do tych ze statusem `'Dostarczone'` LUB `'Wyslane'`, złożonych po `2025-01-01`. Podpowiedź: porównanie z datą w postaci `F.col("data_zamowienia") > "2025-01-01"` działa poprawnie na kolumnie typu daty.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
zamowienia.filter(
    ((F.col("status") == "Dostarczone") | (F.col("status") == "Wyslane"))
    & (F.col("data_zamowienia") > "2025-01-01")
).show(10)
```

</details>

### 📝 Ćwiczenie 3: Nowa kolumna z warunkiem

Dodaj do `pozycje_zamowien` kolumnę `wartosc_pozycji` obliczoną jako `ilosc * cena_jednostkowa * (1 - rabat)`, a potem posortuj malejąco po tej kolumnie i pokaż 5 najdroższych pozycji.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
pozycje_z_wartoscia = pozycje_zamowien.withColumn(
    "wartosc_pozycji", F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))
)
pozycje_z_wartoscia.orderBy(F.col("wartosc_pozycji").desc()).show(5)
```

</details>

### 📝 Ćwiczenie 4: Ile kolumn, ile wierszy

Dla DataFrame'u `pracownicy` wypisz liczbę kolumn (`len(df.columns)`) oraz liczbę wierszy (`count()`), bez konwertowania do pandas.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
print(f"Liczba kolumn: {len(pracownicy.columns)}")
print(f"Liczba wierszy: {pracownicy.count()}")
```

</details>

> 🔥 **Wyzwanie: własny schemat i walidacja**
>
> Stwórz DataFrame `testowi_klienci` z 3 wierszami danych, w których CELOWO jedna wartość `klient_id` jest tekstem (`"abc"`) zamiast liczbą, używając jawnego `StructType` z `IntegerType()` dla `klient_id`. Sprawdź, co się stanie -- czy Spark zgłosi błąd od razu, czy dopiero przy akcji?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schemat_testowy = StructType([
    StructField("klient_id", IntegerType(), True),
    StructField("miasto", StringType(), True),
])

try:
    testowi_klienci = spark.createDataFrame(
        [(1, "Warszawa"), ("abc", "Krakow"), (3, "Gdansk")], schema=schemat_testowy
    )
    testowi_klienci.show()
except Exception as e:
    print(f"Blad: {type(e).__name__}: {e}")
```

Sam createDataFrame() to transformacja (a wlasciwie proces budowy planu) -- w zaleznosci od wersji Sparka i tego, czy dane sa juz lokalnie zmaterializowane, blad walidacji typu moze pojawic sie albo od razu przy tworzeniu, albo dopiero przy pierwszej akcji (show()). To dobra ilustracja leniwej ewaluacji z Modulu 1: bledy tez moga byc 'leniwe'.

</details>

## Co dalej?

W **Module 3** rozwiniemy przekształcenia o agregacje: `groupBy`, funkcje wbudowane
(`pyspark.sql.functions`), obsługę wartości `NULL` i wyrażenia warunkowe
`when`/`otherwise`.